# Model Quality Review

This notebook is designed to run directly inside the project without manual path edits.

It covers:

- loading trained artifacts and datasets
- probabilistic quality metrics for all ranker targets
- threshold-based classification metrics
- calibration diagnostics
- feature importance review
- retrieval similarity diagnostics
- probability distribution inspection

In [ ]:
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root with pyproject.toml and src/")


ROOT = find_project_root(Path.cwd())
sys.path.insert(0, str(ROOT / "src"))

plt.style.use("ggplot")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

ROOT

In [ ]:
TARGETS = ["open", "like", "share", "long_view", "skip_fast", "disengage"]

paths = {
    "ranker": ROOT / "models" / "ranker.joblib",
    "ranker_training": ROOT / "data" / "processed" / "ranker_training.csv",
    "cards": ROOT / "data" / "raw" / "cards.csv",
    "interactions": ROOT / "data" / "raw" / "interactions.csv",
    "users": ROOT / "data" / "raw" / "users.csv",
}

missing = [str(path) for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required artifacts. Run `py scripts/generate_data.py` and `py scripts/train_models.py` first:\n"
        + "\n".join(missing)
    )

ranker_artifact = joblib.load(paths["ranker"])
ranker_training = pd.read_csv(paths["ranker_training"])
cards = pd.read_csv(paths["cards"])
interactions = pd.read_csv(paths["interactions"])
users = pd.read_csv(paths["users"])

print(f"Project root: {ROOT}")
print(f"Ranker training rows: {len(ranker_training):,}")
print(f"Users: {len(users):,} | Cards: {len(cards):,} | Interactions: {len(interactions):,}")
print(f"Targets: {', '.join(TARGETS)}")

In [ ]:
feature_frame = ranker_training.reindex(columns=ranker_artifact["feature_columns"], fill_value=0.0).copy()
scored = ranker_training.copy()

for target, model in ranker_artifact["models"].items():
    probabilities = model.predict_proba(feature_frame)
    scored[f"pred_{target}"] = probabilities[:, 1] if probabilities.shape[1] > 1 else 0.0
    scored[f"label_{target}"] = (scored[f"pred_{target}"] >= 0.5).astype(int)

scored.head()

In [ ]:
def safe_prob_metric(metric_fn, y_true, y_pred):
    try:
        if metric_fn is log_loss:
            return float(metric_fn(y_true, np.clip(y_pred, 1e-6, 1 - 1e-6)))
        return float(metric_fn(y_true, y_pred))
    except ValueError:
        return np.nan


def safe_label_metric(metric_fn, y_true, y_label):
    try:
        return float(metric_fn(y_true, y_label))
    except ValueError:
        return np.nan


probability_rows = []
threshold_rows = []

for target in TARGETS:
    y_true = scored[target].astype(int)
    y_pred = scored[f"pred_{target}"].astype(float)
    y_label = scored[f"label_{target}"].astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_label, labels=[0, 1]).ravel()

    probability_rows.append(
        {
            "target": target,
            "positive_rate": float(y_true.mean()),
            "prediction_mean": float(y_pred.mean()),
            "roc_auc": safe_prob_metric(roc_auc_score, y_true, y_pred),
            "average_precision": safe_prob_metric(average_precision_score, y_true, y_pred),
            "log_loss": safe_prob_metric(log_loss, y_true, y_pred),
            "brier_score": safe_prob_metric(brier_score_loss, y_true, y_pred),
        }
    )
    threshold_rows.append(
        {
            "target": target,
            "accuracy": safe_label_metric(accuracy_score, y_true, y_label),
            "balanced_accuracy": safe_label_metric(balanced_accuracy_score, y_true, y_label),
            "precision": safe_label_metric(precision_score, y_true, y_label),
            "recall": safe_label_metric(recall_score, y_true, y_label),
            "f1": safe_label_metric(f1_score, y_true, y_label),
            "tp": int(tp),
            "fp": int(fp),
            "tn": int(tn),
            "fn": int(fn),
        }
    )

probability_metrics_df = pd.DataFrame(probability_rows).sort_values("roc_auc", ascending=False).reset_index(drop=True)
threshold_metrics_df = pd.DataFrame(threshold_rows).sort_values("f1", ascending=False).reset_index(drop=True)

probability_metrics_df

In [ ]:
threshold_metrics_df

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

probability_metrics_df.plot.bar(x="target", y="roc_auc", ax=axes[0, 0], legend=False, color="#4C78A8")
axes[0, 0].set_title("ROC-AUC by target")
axes[0, 0].set_ylim(0.0, 1.0)
axes[0, 0].tick_params(axis="x", rotation=45)

probability_metrics_df.plot.bar(x="target", y="average_precision", ax=axes[0, 1], legend=False, color="#F58518")
axes[0, 1].set_title("Average precision by target")
axes[0, 1].set_ylim(0.0, 1.0)
axes[0, 1].tick_params(axis="x", rotation=45)

probability_metrics_df.plot.bar(x="target", y="log_loss", ax=axes[1, 0], legend=False, color="#54A24B")
axes[1, 0].set_title("Log loss by target")
axes[1, 0].tick_params(axis="x", rotation=45)

probability_metrics_df.plot.bar(x="target", y="brier_score", ax=axes[1, 1], legend=False, color="#B279A2")
axes[1, 1].set_title("Brier score by target")
axes[1, 1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
target = "like"
calibration_frame = scored[[target, f"pred_{target}"]].copy()
calibration_frame["bin"] = pd.qcut(
    calibration_frame[f"pred_{target}"],
    q=min(10, calibration_frame[f"pred_{target}"].nunique()),
    duplicates="drop",
)
calibration_df = (
    calibration_frame.groupby("bin", observed=False)
    .agg(
        predicted_mean=(f"pred_{target}", "mean"),
        actual_mean=(target, "mean"),
        sample_count=(target, "size"),
    )
    .reset_index()
)
calibration_df["bin"] = calibration_df["bin"].astype(str)
calibration_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(calibration_df["predicted_mean"], calibration_df["actual_mean"], marker="o", label="Model")
axes[0].plot([0, 1], [0, 1], linestyle="--", color="black", label="Ideal")
axes[0].set_title(f"Calibration curve for {target}")
axes[0].set_xlabel("Predicted probability")
axes[0].set_ylabel("Observed rate")
axes[0].legend()

axes[1].hist(scored[f"pred_{target}"], bins=20, color="#72B7B2", edgecolor="black")
axes[1].set_title(f"Prediction distribution for {target}")
axes[1].set_xlabel("Predicted probability")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

## Reading the results

Use the notebook like this:

- higher `roc_auc` and `average_precision` means the target is easier to rank well
- lower `log_loss` and `brier_score` means probabilities are better calibrated
- `accuracy`, `precision`, `recall`, and `f1` show threshold behavior at `0.5`
- calibration closer to the diagonal means predicted probabilities are more trustworthy
- feature importance shows which features drive each target most strongly
- higher retrieval similarity for positive outcomes suggests retrieval is aligned with downstream engagement

In [ ]:
importance_rows = []
for target_name, model in ranker_artifact["models"].items():
    importances = getattr(model, "feature_importances_", None)
    if importances is None:
        continue
    top_pairs = sorted(
        zip(ranker_artifact["feature_columns"], importances),
        key=lambda item: item[1],
        reverse=True,
    )[:12]
    for feature_name, importance in top_pairs:
        importance_rows.append(
            {"target": target_name, "feature": feature_name, "importance": float(importance)}
        )

importance_df = pd.DataFrame(importance_rows)
importance_df.head(20)

In [ ]:
selected_target = "like"
plot_df = importance_df[importance_df["target"] == selected_target].sort_values("importance")

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(plot_df["feature"], plot_df["importance"], color="#E45756")
ax.set_title(f"Top feature importance for {selected_target}")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

In [ ]:
retrieval_rows = []
for target_name in TARGETS:
    grouped = scored.groupby(target_name)["retrieval_similarity"].mean().to_dict()
    retrieval_rows.append(
        {
            "target": target_name,
            "mean_similarity_when_0": float(grouped.get(0, 0.0)),
            "mean_similarity_when_1": float(grouped.get(1, 0.0)),
            "delta": float(grouped.get(1, 0.0) - grouped.get(0, 0.0)),
        }
    )

retrieval_df = pd.DataFrame(retrieval_rows).sort_values("delta", ascending=False).reset_index(drop=True)
retrieval_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(retrieval_df["target"], retrieval_df["delta"], color="#72B7B2")
ax.set_title("Retrieval similarity lift for positive outcomes")
ax.set_xlabel("Mean similarity(positive) - Mean similarity(negative)")
plt.tight_layout()
plt.show()

In [ ]:
selected_target = "like"
y_true = scored[selected_target].astype(int)
y_label = scored[f"label_{selected_target}"].astype(int)
cm = confusion_matrix(y_true, y_label, labels=[0, 1])
cm_normalized = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(5, 5))
image = ax.imshow(cm_normalized, cmap="Blues")
ax.set_title(f"Normalized confusion matrix for {selected_target}")
ax.set_xticks([0, 1], labels=["pred 0", "pred 1"])
ax.set_yticks([0, 1], labels=["true 0", "true 1"])

for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm_normalized[i, j]:.2f}\n({cm[i, j]})", ha="center", va="center", color="black")

fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()